In [1]:
%pip install groq python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import getpass

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter Groq API Key:")

In [8]:
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

ROUTER_MODEL = "llama-3.1-8b-instant"
EXPERT_MODEL = "llama-3.3-70b-versatile"

In [3]:
MODEL_CONFIG = {
    "technical": {
        "system_prompt": """You are a Technical Support Expert. Be precise, logical and code-focused. Provide debugging steps and example fixes."""
    },
    "billing": {
        "system_prompt": """You are a Billing Support Specialist. Be empathetic and policy-driven. Explain charges clearly and outline refund steps."""
    },
    "general": {
        "system_prompt": """You are a helpful Customer Support Assistant. Handle general inquiries professionally."""
    },
    "tool": {
        "system_prompt": """You are a tool routing expert."""
    }
}

In [4]:
def route_prompt(user_input):
    l_input = user_input.lower()
    if "price" in l_input and "bitcoin" in l_input:
        return "tool"
    
    routing_prompt = f"""
Classify the following text into one of these categories, [technical, billing, general]

Return ONLY the category name, no punctuation or pronouns.

Text: {user_input}
"""
    
    response = client.chat.completions.create(
        model=ROUTER_MODEL,
        messages=[
            {"role": "system", "content": "You are a strict classification engine."},
            {"role": "user", "content": routing_prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip().lower()

In [ ]:
def get_bitcoin_price():
    return "$65,432.10 (mock price)"

In [6]:
def process_request(user_input):
    category = route_prompt(user_input)

    if category not in MODEL_CONFIG:
        category = "general"
    
    if category == "tool":
        if "bitcoin" in user_input.lower():
            price = get_bitcoin_price()
            return f"[Routed to: TOOL]\n\nCurrent Bitcoin price is {price}"
        else:
            return "[Routed to: TOOL]\n\nRequested live data."
    
    system_prompt = MODEL_CONFIG[category]["system_prompt"]

    response = client.chat.completions.create(
        model=EXPERT_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ],
        temperature=0.7
    )

    return f"[Routed to: {category.upper()}]\n\n{response.choices[0].message.content}"

In [9]:
query = "My python script is throwing an IndexError on line 5."
print(process_request(query))

[Routed to: TECHNICAL]

To assist you with debugging the IndexError on line 5 of your Python script, I'll need more information. However, I can guide you through a general approach to troubleshooting this issue.

### Steps to Debug:

1. **Identify the Line:** Confirm that the error is indeed on line 5 and understand what operation is being performed on that line.
2. **Understand IndexError:** An `IndexError` occurs when you try to access an element in a list (or other sequence type) using an index that does not exist. For example, trying to access the 5th element (`my_list[4]`) in a list that only has 4 elements (`my_list = [1, 2, 3, 4]`).
3. **Check Data:** Before the line causing the error, print out or inspect the data you're trying to access. This will help you understand the structure and length of the data.
4. **Validate Index:** Ensure that the index you're using to access the data is within the valid range. Remember, Python uses 0-based indexing.

### Example Fix:

Let's say yo

In [10]:
print(process_request("I was charged twice for my subscription."))

[Routed to: BILLING]

I'm so sorry to hear that you were charged twice for your subscription. I can imagine how frustrating that must be for you. I'm here to help you resolve this issue as quickly as possible.

First, I want to assure you that we take situations like this very seriously and will do our best to rectify the error. I'll need to investigate this further to understand what happened.

Can you please provide me with your subscription details, such as your account name, subscription plan, and the dates of the duplicate charges? This will help me to look into the matter and determine the cause of the error.

Once I've investigated, if we confirm that the duplicate charge was an error on our part, I'll be happy to process a refund for the incorrect charge. Please note that our refund policy states that we will refund any incorrect charges within 5-7 business days of the request.

To initiate the refund process, I'll need to verify some information with you. If everything checks 

In [11]:
print(process_request("Hi, how are you?"))

[Routed to: GENERAL]

I'm doing well, thank you for asking. How can I assist you today? Do you have any questions or concerns that I can help with? I'm here to provide you with the best possible support.


In [12]:
print(process_request("What is the current price of Bitcoin?"))

[Routed to: TOOL]

Current Bitcoin price is $65,432.10
